# Practical: GANs, Wasserstein GANs and spectral normalization in 2-D

This practical follows the lecture. We work on a 2-D toy problem, so that we can **see** what the networks do and **measure** things exactly.

1. **Standard GAN and mode collapse**: train a standard GAN on a mixture of two Gaussians with unequal weights (the same as in the score-matching practical), and see what it does with the small mode.
2. **Discriminator vs critic**: compare, in 1-D, the optimal discriminator of a standard GAN with the 1-Lipschitz critic of a Wasserstein GAN.
3. **Spectral normalization**: implement it, and check that it makes the critic 1-Lipschitz.
4. **Why GAN training oscillates**: the simplest GAN, reduced to two numbers.
5. **Going further**: DCGAN on MNIST, following the PyTorch tutorial.

Sections 0 to 3 are the core of the practical. **Sections 4 and 5 are for those who finish early**: they can be done in any order.

Cells marked **Exercise** contain a `TODO` for you to fill in. Each one is followed by a **check** cell: run it, and it either prints `OK` or tells you what is wrong.

*Requirements:* `torch`, `numpy`, `scipy`, `matplotlib`, and the file **`utils_gan.py`**, which contains the helpers: the data, the measurements, the plots and the checks. You do not need to read it. Keep it in the same folder as this notebook; on Google Colab, the first cell downloads it. Everything runs on a CPU; the training in Section 1 takes about a minute.

In [ ]:
import math
import os
import urllib.request

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# The helpers are in utils_gan.py. If it is not next to this notebook (e.g. on Colab), download it.
UTILS_URL = "https://raw.githubusercontent.com/franciscoandrade90/deep-learning-practicals/main/gans/utils_gan.py"
if not os.path.exists("utils_gan.py"):
    urllib.request.urlretrieve(UTILS_URL, "utils_gan.py")
from utils_gan import *

torch.manual_seed(0)

## 0. Data, networks and tools

**The data**: the mixture of the score-matching practical,
$$
p(\mathbf{x}) = 0.2\,\mathcal{N}\big(\mathbf{x};(-5,-5),\mathbf{I}\big) + 0.8\,\mathcal{N}\big(\mathbf{x};(5,5),\mathbf{I}\big).
$$
Each Gaussian is a *mode*: a good generator must produce both, with 20% of its samples in the small one.

In [ ]:
x_data = sample_data()                                              # 10,000 training samples
fig, ax = plt.subplots(figsize=(5, 5))
plot_real(x_data[:2000], ax)
ax.set_title("data: 20% around (-5, -5), 80% around (5, 5)")
plt.show()

**The networks.** Both networks are small multilayer perceptrons (MLPs): two hidden layers of `HIDDEN = 128` units with ReLU activations.

- **The generator** $G$ maps a latent $\mathbf{z} \in \mathbb{R}^2$ (`Z_DIM = 2`), drawn from $\mathcal{N}(0, \mathbf{I})$, to a point $G(\mathbf{z}) \in \mathbb{R}^2$:
$$
\mathbb{R}^2 \xrightarrow{\text{linear}} \mathbb{R}^{128} \xrightarrow{\text{ReLU}} \xrightarrow{\text{linear}} \mathbb{R}^{128} \xrightarrow{\text{ReLU}} \xrightarrow{\text{linear}} \mathbb{R}^2 .
$$
There is **no activation after the last layer**: a generated point can be anywhere in the plane, e.g. near $(-5,-5)$.

- **The discriminator** (or critic) $D$ maps a point $\mathbf{x} \in \mathbb{R}^{d_\text{in}}$ to a single number:
$$
\mathbb{R}^{d_\text{in}} \xrightarrow{\text{linear}} \mathbb{R}^{128} \xrightarrow{\text{ReLU}} \xrightarrow{\text{linear}} \mathbb{R}^{128} \xrightarrow{\text{ReLU}} \xrightarrow{\text{linear}} \mathbb{R} .
$$
Here too there is **no activation after the last layer**. For the standard GAN, $D(\mathbf{x})$ is a probability, so the network should end with a sigmoid. We put the sigmoid in the loss instead (Exercise 2), for numerical reasons: the network outputs a raw score $t$, and $D(\mathbf{x}) = \sigma(t)$. For the WGAN, the critic's output is used directly.

Two arguments of `build_critic` will be used later in the notebook:

- `d_in` is the input dimension: 2 here, 1 for the 1-D experiments of Section 2.
- `linear` is the *type* of linear layer: `nn.Linear` by default, a spectrally normalized layer in Sections 2 and 3. So the critic must build **all three** of its linear layers with `linear(n_in, n_out)`, never with `nn.Linear` directly.

**Exercise 1.** Write `build_generator()` and `build_critic(linear, d_in)`, which return the two networks as `nn.Sequential` models.

*Hint:* `nn.Sequential(nn.Linear(a, b), nn.ReLU(), ...)` chains layers. The generator is the same as the critic with `Z_DIM` inputs and 2 outputs.

In [ ]:
Z_DIM, HIDDEN = 2, 128


def build_generator():
    '''Generator: (n, Z_DIM) -> (n, 2).'''
    # TODO
    raise NotImplementedError


def build_critic(linear=nn.Linear, d_in=2):
    '''Discriminator / critic: (n, d_in) -> (n, 1). `linear` is the type of linear layer.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_networks(build_generator, build_critic, Z_DIM, HIDDEN)

**Three measurements** (in `utils_gan.py`), which a real application would not have, but our toy problem allows:

- `fraction_small_mode(x)`: the fraction of the samples `x` on the side of the small mode. As in the score-matching practical, the two modes are separated by the line $x_1 + x_2 = 0$.
- `wasserstein(x, y)`: the **exact** Wasserstein distance between two sets of points of the same size. Between two empirical distributions with $n$ points each, the optimal transport plan is a matching of the points, which `scipy` finds exactly.
- `wasserstein_large(x_real, x_gen)`: the same distance, but keeping only the points on the side of the **large mode**, with the same number of points on each side. It tells us how well the generator fits the large mode alone.

All measurements use 1000 fixed real samples, `X_EVAL`.

In [ ]:
print_floors()

**The training loop** is written for any GAN loss. At each step, the critic is updated `n_critic` times, then the generator once. What changes between methods is only:

- `losses(D, x_real, x_fake)`, which returns `(loss_D, loss_G)`,
- the type of linear layers in the critic.

Both networks are trained with the Adam optimiser, with the same settings as in the DCGAN tutorial of Section 5. Every `record_every` steps, `record` (from `utils_gan.py`) measures the generator on 1000 samples.

In [ ]:
def train(losses, linear=nn.Linear, n_steps=6000, n_critic=5, batch_size=128, lr=2e-4,
          seed=0, record_every=500):
    '''Trains a generator and a critic on x_data. Returns (G, D, history).'''
    torch.manual_seed(seed)
    G, D = build_generator(), build_critic(linear)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    history = new_history()

    for step in range(1, n_steps + 1):
        # ---- critic / discriminator updates
        for _ in range(n_critic):
            x_real = x_data[torch.randint(len(x_data), (batch_size,))]
            with torch.no_grad():
                x_fake = G(torch.randn(batch_size, Z_DIM))
            loss_D, _ = losses(D, x_real, x_fake)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

        # ---- generator update
        x_real = x_data[torch.randint(len(x_data), (batch_size,))]
        x_fake = G(torch.randn(batch_size, Z_DIM))
        _, loss_G = losses(D, x_real, x_fake)
        opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        # ---- measurements
        if step % record_every == 0:
            record(history, step, G, D, losses, Z_DIM, n_steps)
    return G, D, history

## 1. Standard GAN and mode collapse

The losses of the lecture are:
$$
\mathcal{L}_D = -\,\mathbb{E}_{\mathbf{x}\sim\mu}\big[\log D(\mathbf{x})\big] - \mathbb{E}_{\mathbf{x}\sim\mu_G}\big[\log\big(1 - D(\mathbf{x})\big)\big],
\qquad
\mathcal{L}_G = \mathbb{E}_{\mathbf{x}\sim\mu_G}\big[\log\big(1 - D(\mathbf{x})\big)\big],
$$
where $\mathcal{L}_G$ is the original **minimax** (or *saturating*) generator loss: the generator minimizes the same quantity that the discriminator maximizes. In Section 2, we compare it with the **non-saturating** loss $-\mathbb{E}_{\mathbf{x}\sim\mu_G}[\log D(\mathbf{x})]$, which is the one used in practice.

**Exercise 2.** Write `gan_losses(D, x_real, x_fake)`, which returns `(loss_D, loss_G)`, both single numbers.

*Hint:* the network `D` outputs raw scores `t = D(x)`, of shape `(n, 1)`, and the probability is $\sigma(t)$. Use `F.logsigmoid(t)` for $\log \sigma(t)$, and `F.logsigmoid(-t)` for $\log(1 - \sigma(t))$, since $1 - \sigma(t) = \sigma(-t)$. Do not compute `torch.log(torch.sigmoid(t))`: it gives $-\infty$ as soon as $\sigma(t)$ rounds to 0.

In [ ]:
def gan_losses(D, x_real, x_fake):
    '''D: (n, 2) -> (n, 1) raw scores; x_real, x_fake: (n, 2). Returns (loss_D, loss_G), two scalars.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_gan_losses(gan_losses)

Now train the standard GAN (this takes about a minute):

In [ ]:
G_gan, D_gan, hist_gan = train(gan_losses)
plot_snapshots(hist_gan, "standard GAN")
plot_curves({"standard GAN": hist_gan})

**Questions.**

1. What fraction of the generated samples is in the small mode, compared with the 20% of the data?
2. Compare the generator loss and the two Wasserstein distances (all data, and large mode only) during training. What explains the value of the first one? Can the generator loss tell you whether the samples are getting better?
3. Write $\mu = 0.2\,\varphi_\text{small} + 0.8\,\varphi_\text{large}$ for the data, where $\varphi_\text{small}, \varphi_\text{large}$ are its two Gaussians. Assume that the generator is a Gaussian centred at $(5, 5)$ with identity covariance: $\mu_G = \varphi_\text{large}$. What is the best discriminator $D^*$ against it, in particular at the fake samples? What is the generator loss $\mathbb{E}_{\mathbf{x} \sim \mu_G}\big[\log\big(1 - D^*(\mathbf{x})\big)\big]$ then? Compare with the values that the generator loss of the standard GAN reaches during training. (The dashed line, $\log(1/2) \approx -0.69$, is its value for a perfect generator, $\mu_G = \mu$, against which $D^* = 1/2$ everywhere.)
4. Each generator step comes after 5 discriminator steps, so the discriminator is close to optimal. Why could a *better* discriminator make things *worse* for the generator? *(This is what Section 2 is about.)*

**Is it bad luck?** GAN training is random: the initialisation, the noise $z$ and the minibatches change from one run to the next, and so do the exact numbers on different machines. Here is what 14 runs of exactly the same training gave on our machine (seeds 0 to 13):

| Outcome | Runs |
|---|---|
| collapse onto the large mode (0% in the small mode), as above | 11 |
| collapse onto the small mode (100% in the small mode) | 1 |
| both modes, but with too few samples in the small one (12% and 17%), and a thin line of samples between the modes | 2 |

So mode collapse is not bad luck: it is what usually happens. Even when both modes are found, the proportions are wrong. The thin line between the modes cannot be avoided: the generator is a continuous function of the Gaussian noise $z$, so its samples cannot form two separate clouds.

*Your own run may give any of these outcomes. The questions below assume the most common one, a collapse onto the large mode; if yours is different, compare with your neighbours, or train again with another seed: `train(gan_losses, seed=1)` (about a minute).*

**Link with the score-matching practical.** Both practicals use the same mixture, so we can compare the samplers on the same question: what fraction of the samples lands in the small mode (target: 20%)?

| Method | Small mode |
|---|---|
| Langevin dynamics (score matching) | about 50%: wrong proportions |
| Annealed Langevin dynamics (score matching) | about 20% |
| Standard GAN | usually 0%: the mode is dropped |

## 2. Discriminator vs critic

The **WGAN** losses of the lecture are, for a critic $f$ that must be 1-Lipschitz:
$$
\mathcal{L}_f = -\big(\mathbb{E}_{\mathbf{x}\sim\mu}[f(\mathbf{x})] - \mathbb{E}_{\mathbf{x}\sim\mu_G}[f(\mathbf{x})]\big),
\qquad
\mathcal{L}_G = -\,\mathbb{E}_{\mathbf{x}\sim\mu_G}[f(\mathbf{x})].
$$
The critic maximises the gap between real and fake; the generator tries to raise the critic's value on its samples.

**Exercise 3.** Write `wgan_losses(f, x_real, x_fake)`, which returns `(loss_f, loss_G)`.

In [ ]:
def wgan_losses(f, x_real, x_fake):
    '''f: (n, d) -> (n, 1); x_real, x_fake: (n, d). Returns (loss_f, loss_G), two scalars.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_wgan_losses(wgan_losses)

The critic must be 1-Lipschitz. In dimension 1, this means that its derivative is bounded by 1: $|f'(x)| \le 1$, i.e. the critic can never be steeper than a slope of 1. For now, we use PyTorch's **spectral normalization** as a black box: each linear layer of the critic is wrapped so that it is 1-Lipschitz. In Section 3 you will implement it yourself.

In 1-D, we train a discriminator and a critic to separate real samples around $-2$ from fake samples around $+2$ (the fakes do not move), and look at what they learned.

In [ ]:
def sn_linear(d_in, d_out):
    '''A linear layer made 1-Lipschitz by PyTorch's spectral normalization (see Section 3).'''
    return nn.utils.parametrizations.spectral_norm(nn.Linear(d_in, d_out))

In [ ]:
def fit_1d(losses, linear=nn.Linear, n_steps=3000, seed=0):
    '''Trains a 1-D discriminator / critic to separate real_1d (around -2) from fake_1d (around +2).'''
    torch.manual_seed(seed)
    f = build_critic(linear, d_in=1)
    opt = torch.optim.Adam(f.parameters(), lr=5e-4, betas=(0.5, 0.999))
    for _ in range(n_steps):
        loss, _ = losses(f, real_1d(256), fake_1d(256))
        opt.zero_grad(); loss.backward(); opt.step()
    return f


D_1d = fit_1d(gan_losses)
f_1d = fit_1d(wgan_losses, linear=sn_linear)
plot_1d(D_1d, f_1d)

**How much does the generator move?** The generator learns only through the chain rule. Here everything is 1-D: a fake sample $x = G_\theta(z)$ is a number, and $D'(x)$ is the ordinary derivative of $D$. For the three generator losses:

- standard GAN, **minimax** (saturating) loss (the one used in Section 1):
$$
\mathcal{L}_G = \mathbb{E}\big[\log\big(1 - D(G_\theta(z))\big)\big]
\quad\Longrightarrow\quad
\frac{\partial \mathcal{L}_G}{\partial \theta} = \mathbb{E}\Big[-\frac{D'(x)}{1 - D(x)}\;\frac{\partial x}{\partial \theta}\Big]
$$
- standard GAN, **non-saturating** loss (the one used in practice, e.g. in the DCGAN tutorial of Section 5):
$$
\mathcal{L}_G = -\,\mathbb{E}\big[\log D(G_\theta(z))\big]
\quad\Longrightarrow\quad
\frac{\partial \mathcal{L}_G}{\partial \theta} = \mathbb{E}\Big[-\frac{D'(x)}{D(x)}\;\frac{\partial x}{\partial \theta}\Big]
$$
- **WGAN**:
$$
\mathcal{L}_G = -\,\mathbb{E}\big[f(G_\theta(z))\big]
\quad\Longrightarrow\quad
\frac{\partial \mathcal{L}_G}{\partial \theta} = \mathbb{E}\Big[-f'(x)\;\frac{\partial x}{\partial \theta}\Big]
$$

In each case, the derivative of the discriminator (or critic) at the fake samples appears, multiplied by a factor that depends on the loss. We call the term in front of $\partial x / \partial \theta$ (with a minus sign) the **push** on a fake sample: for example $D'(x) / D(x)$ for the non-saturating loss. The first two losses use the **same** discriminator `D_1d`. Below, we compute the push at the fakes around $+2$.

In [ ]:
x = fake_1d(1000).requires_grad_(True)
generator_losses = {
    "standard GAN, minimax loss       ": F.logsigmoid(-D_1d(x)),    # log(1 - D)
    "standard GAN, non-saturating loss": -F.logsigmoid(D_1d(x)),    # -log D
    "WGAN                             ": -f_1d(x),
}
print("average push on the fake samples (negative = towards the real data at -2):")
for name, loss in generator_losses.items():
    push = -torch.autograd.grad(loss.sum(), x)[0].mean().item()
    print(f"  {name}  {push:+.2g}")

**Questions.**

1. In the figure, where is the derivative of the discriminator $D$ non-zero? Is it non-zero at the fake samples?
2. Look at the pushes. Using the formulas above, explain why the generator is stuck with the minimax loss, and why the non-saturating loss still pushes the fakes. Is the size of that push something we control?
3. Explain the push for the WGAN.
4. Compare with the figure of Arjovsky et al. in the lecture.

## 3. Spectral normalization

Spectral normalization (Miyato et al., 2018) divides each weight matrix $W$ by its largest singular value $\sigma(W)$, so that each linear layer is 1-Lipschitz. Since ReLU is 1-Lipschitz too, and the Lipschitz constant of a composition is at most the product of the constants, the whole critic is then 1-Lipschitz.

### 3.1 Power iteration

The largest singular value is computed by power iteration, as in the lecture: starting from a random vector $\mathbf{u}$, repeat
$$
\mathbf{v} \leftarrow \frac{W^\top \mathbf{u}}{\|W^\top \mathbf{u}\|},
\qquad
\mathbf{u} \leftarrow \frac{W \mathbf{v}}{\|W \mathbf{v}\|},
\qquad\text{then}\qquad
\sigma(W) \approx \mathbf{u}^\top W \mathbf{v}.
$$

**Exercise 4.** Write `power_iteration(W, u, n_iter)`, which starts from the vector $\mathbf{u}$, repeats the two updates above `n_iter` times, and returns the final vectors $\mathbf{u}$ and $\mathbf{v}$. The estimate of $\sigma(W)$ is then $\mathbf{u}^\top W \mathbf{v}$: we compute it outside the function, because in Section 3.2 we will need it to be part of the gradient, while the updates of $\mathbf{u}$ and $\mathbf{v}$ must not be.

*Hints:* `F.normalize(v, dim=0)` divides a vector by its norm. Do the updates under `with torch.no_grad():`. You do not need an initial $\mathbf{v}$: the first update computes it from $\mathbf{u}$.

In [ ]:
def power_iteration(W, u, n_iter):
    '''W: (m, n); u: (m,), the starting vector. Runs n_iter steps of power iteration (without gradient).
    Returns the final u, of shape (m,), and v, of shape (n,).'''
    # TODO
    raise NotImplementedError

In [ ]:
check_power_iteration(power_iteration)

**Exercise 5.** For a random $15 \times 15$ matrix, compute the largest singular value with PyTorch, `torch.linalg.svdvals(W)` (it returns the singular values in decreasing order), and with your `power_iteration`: 10 iterations from a random $\mathbf{u}$ (`torch.randn(15)`), then $\mathbf{u}^\top W \mathbf{v}$. Compare the two.

In [ ]:
torch.manual_seed(0)
W = torch.randn(15, 15)
# TODO
raise NotImplementedError
print(f"torch.linalg.svdvals: {exact:.4f}    power iteration (10 iterations): {estimate:.4f}")

### 3.2 A spectrally normalized linear layer

In practice, we do not run 20 iterations at every step: the layer keeps its vector $\mathbf{u}$ from one forward pass to the next, and does **one** iteration per pass. Since $W$ changes slowly during training, this is enough to track $\sigma(W)$. This is what `sn_linear` did in Section 2.

**Exercise 6.** Complete `SNLinear`, a linear layer that uses $W / \sigma(W)$ instead of $W$.

*Hint:* fill in the three `...` in `forward`, following the comments. `F.linear(x, weight, bias)` applies a linear layer with the given weight.

In [ ]:
class SNLinear(nn.Linear):
    '''nn.Linear whose weight is divided by its largest singular value.'''

    def __init__(self, d_in, d_out):
        super().__init__(d_in, d_out)
        self.register_buffer("u", F.normalize(torch.randn(d_out), dim=0))   # kept between passes

    def forward(self, x):
        '''x: (n, d_in) -> (n, d_out).'''
        # 1. One step of power iteration, starting from the u stored in the layer
        u, v = ...
        # 2. Store the new u in the layer, for the next forward pass
        self.u = u
        # 3. The estimate of sigma(W); it must be part of the gradient
        sigma = ...
        # 4. The linear layer, with the normalized weight
        return F.linear(x, ..., self.bias)

In [ ]:
check_SNLinear(SNLinear)

### 3.3 Checking the Lipschitz bound

For a differentiable critic, the Lipschitz constant is the largest norm of its gradient. We estimate it on a grid of points.

**Exercise 7.** Write `max_grad_norm(f, points)`, which returns $\max_i \|\nabla f(\mathbf{x}_i)\|$ over the points.

*Hint:* make a copy of `points` with `requires_grad_(True)`, then use `torch.autograd.grad(f(x).sum(), x)`; the gradient has the same shape as `x`.

In [ ]:
def max_grad_norm(f, points):
    '''f: (n, 2) -> (n, 1); points: (n, 2). Returns the largest gradient norm, a number.'''
    # TODO
    raise NotImplementedError

In [ ]:
check_max_grad_norm(max_grad_norm)

Now we use **your** `SNLinear` in the 1-D example of Section 2. The true Wasserstein distance between the two 1-D distributions is 4 (the real distribution is the fake one shifted by 4).

In [ ]:
f_my_1d = fit_1d(wgan_losses, linear=SNLinear)
f_my_1d.eval()
gap = (f_my_1d(real_1d(20_000)).mean() - f_my_1d(fake_1d(20_000)).mean()).item()
print(f"critic gap: {gap:.3f}   (true Wasserstein distance: 4)")

plot_1d_critic(f_my_1d, "critic with your SNLinear")

Finally, we compare this critic with the discriminator of the standard GAN trained in Section 1, which has no constraint. For each, we print its measured Lipschitz constant (largest gradient norm on a grid), and the upper bound given by the product of the spectral norms of its layers.

In [ ]:
compare_lipschitz(D_gan, f_my_1d, max_grad_norm, SNLinear)

**Questions.**

1. In 1-D, what gap does the critic find? What does its graph look like?
2. Is the critic 1-Lipschitz? And the discriminator of the standard GAN?
3. The table also gives the product of the spectral norms of the layers. Check that it is always at least the largest gradient norm. For which of the two networks is it close?

## 4. Why GAN training oscillates (if you finish early)

In Section 1, the training did not settle: the curves kept moving. Here is the simplest GAN that shows why.

- **Data:** 1000 samples from $\mathcal{N}(0, 1)$.
- **Generator:** $G_\theta(z) = \theta + z$, with $z \sim \mathcal{N}(0, 1)$: it shifts the noise by $\theta$. It matches the data when $\theta \approx 0$.
- **Critic:** $D(x) = \psi\,x$, a line through the origin with slope $\psi$.

No autograd is needed here, so we use NumPy. We train it as in Section 1, with the WGAN losses: the critic does 5 steps, then the generator does one step. As in a WGAN, the critic must be 1-Lipschitz, so we clip its weight: $\psi \in [-1, 1]$. With these losses, the gradients are simple:

- critic: the mean of the fake samples minus the mean of the real samples;
- generator: $-\psi$.

**Exercise 8.** Complete `train_toy_gan`: fill in the two `...`, following the comments.

In [ ]:
def train_toy_gan(x, z, h=0.1, n_steps=500, n_critic=5):
    '''x: real samples, z: noise values (NumPy arrays). Trains the GAN from theta = 1, psi = 0, with step size h.
    Returns the values of theta and psi over the steps.'''
    theta, psi = 1.0, 0.0                       # the fakes are shifted by 1, and the critic is flat
    thetas, psis = [theta], [psi]
    for step in range(n_steps):
        # ---- the critic: n_critic gradient steps on psi
        for _ in range(n_critic):
            # its gradient is the mean of the fakes (theta + z) minus the mean of the real samples x
            psi = ...
            psi = max(-1.0, min(1.0, psi))      # weight clipping: keeps the critic 1-Lipschitz
        # ---- the generator: one gradient step on theta, with the new psi
        # its gradient is -psi
        theta = ...
        thetas.append(theta)
        psis.append(psi)
    return np.array(thetas), np.array(psis)

In [ ]:
check_train_toy_gan(train_toy_gan)

In [ ]:
rng = np.random.default_rng(0)
x, z = rng.standard_normal(1000), rng.standard_normal(1000)    # real samples and noise values
plot_toy_gan({"WGAN": train_toy_gan(x, z)}, equilibrium=x.mean() - z.mean())

**Question.** Does the generator settle? Explain in words why it *overshoots* the equilibrium. *(Hint: what is the critic's slope when the generator reaches it?)*

## 5. Going further: DCGAN on MNIST (if you finish early)

The [DCGAN tutorial of PyTorch](https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html) trains the DCGAN of the lecture on faces (CelebA). You can run it on MNIST with three changes:

1. Replace the dataset by MNIST, resized to 64×64 so that the tutorial's architecture works unchanged:
   ```python
   dataset = dset.MNIST(root="data", download=True,
                        transform=transforms.Compose([
                            transforms.Resize(64),
                            transforms.ToTensor(),
                            transforms.Normalize((0.5,), (0.5,)),
                        ]))
   ```
2. Set the number of channels to `nc = 1` (grey-level images).
3. Use a GPU: on Google Colab, *Runtime → Change runtime type → GPU*. On a CPU, one epoch takes a long time.

**Things to look at:**

- How do the samples evolve over the epochs? Compare with the bedrooms of the lecture.
- Are all 10 digits present in a grid of 64 samples? Does it change over training?
- Plot the losses of $G$ and $D$: do they tell you when the samples are good? (The tutorial uses the **non-saturating** generator loss: for the generator update, it labels the fake samples as real. See Section 2 for why.)
- Add spectral normalization to the discriminator: wrap each `nn.Conv2d` of `Discriminator` in `nn.utils.parametrizations.spectral_norm`, and remove its `nn.BatchNorm2d` layers (Miyato et al. do not use batch normalization in the discriminator). What changes?

**More 2-D experiments (optional):**

- **A weaker discriminator.** Train the standard GAN with `n_critic=1`. Does it find the small mode? How does this relate to Section 2?
- **A WGAN on the mixture.** Train `train(wgan_losses, linear=sn_linear)`, about two minutes. What fraction of its samples lands in the small mode? Look at the shape of the two modes too.